In [ ]:
from IPython.display import display, HTML
from pathlib import Path

from hydra import initialize, compose
from hydra.core.global_hydra import GlobalHydra

from lensieve.image import load_image
from lensieve.retrieval.search import search_images
from lensieve.retrieval.schema import SearchArgs
from lensieve.models.model_manager import get_model_manager
from lensieve.data_store import DataStore

In [ ]:
GlobalHydra.instance().clear()

with initialize(config_path="../configs", version_base=None):
    cfg = compose(config_name="config")

model_manager = get_model_manager(cfg)
data_store = DataStore("../data/large_samsung")


In [ ]:
args = SearchArgs(text_query="apartment renovations", max_results=30)

results = search_images(args, model_manager, data_store)

In [ ]:
# parameters
MAX_WIDTH = 200   # px
IMAGES_PER_ROW = 5

def render_images(res):
    rows = []
    current_row = []

    for i, row in enumerate(res):
        path = Path(row.path)
        score = row.score

        try:
            img = load_image(path)

            # resize (keep aspect ratio)
            w, h = img.size
            scale = MAX_WIDTH / w
            img = img.resize((int(w * scale), int(h * scale)))

            # convert to base64 for inline display
            import io, base64
            buffer = io.BytesIO()
            img.save(buffer, format="JPEG")
            img_b64 = base64.b64encode(buffer.getvalue()).decode()

            html = f"""
            <div style="text-align:center; margin:5px;">
                <img src="data:image/jpeg;base64,{img_b64}" />
                <div>{score:.4f}</div>
            </div>
            """
            current_row.append(html)

        except Exception as e:
            current_row.append(f"<div>Error: {e}</div>")

        if len(current_row) == IMAGES_PER_ROW:
            rows.append(current_row)
            current_row = []

    if current_row:
        rows.append(current_row)

    # build table
    table_html = "<table>"
    for row in rows:
        table_html += "<tr>" + "".join(f"<td>{cell}</td>" for cell in row) + "</tr>"
    table_html += "</table>"

    display(HTML(table_html))

render_images(results.hits)